In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load in 

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
from scipy.sparse import hstack
from sklearn.metrics import roc_auc_score, accuracy_score, log_loss
from sklearn.preprocessing import MinMaxScaler

from tqdm import tqdm_notebook, tqdm
from scipy import stats

# Input data files are available in the "../input/" directory.
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
import gc
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Any results you write to the current directory are saved as output.


/kaggle/input/test.csv.zip
/kaggle/input/test.csv
/kaggle/input/sample_submission.csv.zip
/kaggle/input/train.csv.zip
/kaggle/input/description.md
/kaggle/input/sample_submission.csv
/kaggle/input/train.csv
/kaggle/input/google-quest-challenge/test.csv.zip
/kaggle/input/google-quest-challenge/test.csv
/kaggle/input/google-quest-challenge/sample_submission.csv.zip
/kaggle/input/google-quest-challenge/train.csv.zip
/kaggle/input/google-quest-challenge/description.md
/kaggle/input/google-quest-challenge/sample_submission.csv
/kaggle/input/google-quest-challenge/train.csv


In [2]:
def spearman_corr(y_true, y_pred):
        if np.ndim(y_pred) == 2:
            corr = np.mean([stats.spearmanr(y_true[:, i], y_pred[:, i])[0] for i in range(y_true.shape[1])])
        else:
            corr = stats.spearmanr(y_true, y_pred)[0]
        return corr


In [3]:
train = pd.read_csv('../input/google-quest-challenge/train.csv').fillna(' ')
test = pd.read_csv('../input/google-quest-challenge/test.csv').fillna(' ')
train.head()


,qa_id,question_title,question_body,question_user_name,question_user_page,answer,answer_user_name,answer_user_page,url,category,...,question_well_written,answer_helpful,answer_level_of_information,answer_plausible,answer_relevance,answer_satisfaction,answer_type_instructions,answer_type_procedure,answer_type_reason_explanation,answer_well_written
0,9622,Which parts of fresh Fenugreek am I supposed t...,The fresh Fenugreek which I bought contains:\n...,Aquarius_Girl,https://cooking.stackexchange.com/users/6168,I would just pull off all the little stems wit...,spiceyokooko,https://cooking.stackexchange.com/users/14539,http://cooking.stackexchange.com/questions/292...,LIFE_ARTS,...,1.000000,1.000000,0.666667,1.000000,1.000000,1.000000,0.666667,0.333333,0.000000,1.000000
1,3515,Is decoherence even possible in anti de Sitter...,Is decoherence even possible in anti de Sitter...,theorist,https://physics.stackexchange.com/users/6788,"Your question is not about AdS at all, it is a...",Ron Maimon,https://physics.stackexchange.com/users/4864,http://physics.stackexchange.com/questions/185...,SCIENCE,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,1.000000
2,2012,How to show the integers have same cardinality...,How would I show the following have a bijectio...,Fernando Martinez,https://math.stackexchange.com/users/37244,"I like the one that snake around, $0, -1, 1, -...",Adam Hughes,https://math.stackexchange.com/users/58831,http://math.stackexchange.com/questions/873927...,SCIENCE,...,0.777778,0.888889,0.666667,1.000000,0.888889,0.866667,0.666667,0.666667,0.333333,0.777778
3,5460,how to pass value from visual force page (Inpu...,I have created a Vf page from which i need to ...,jack,https://salesforce.stackexchange.com/users/8706,"When you submit the page, Visualforce will upd...",Keith C,https://salesforce.stackexchange.com/users/887,http://salesforce.stackexchange.com/questions/...,TECHNOLOGY,...,0.888889,1.000000,0.666667,1.000000,1.000000,0.933333,1.000000,0.000000,0.000000,0.888889
4,2046,How to set precision and format labeled Slider...,"For my calculations, I need increased precisio...",Cendo,https://mathematica.stackexchange.com/users/1290,"Two, now three, ways: I think I'd recommend t...",Michael E2,https://mathematica.stackexchange.com/users/4999,http://mathematica.stackexchange.com/questions...,TECHNOLOGY,...,1.000000,0.888889,0.833333,0.888889,1.000000,1.000000,1.000000,0.000000,0.000000,1.000000


In [4]:
train.shape


(5471, 41)

In [5]:
test.shape


(608, 11)

In [6]:
np.unique(train['category'].values)


array(['CULTURE', 'LIFE_ARTS', 'SCIENCE', 'STACKOVERFLOW', 'TECHNOLOGY'],
      dtype=object)

In [7]:
train_text_1 = train['question_body']
test_text_1 = test['question_body']
all_text_1 = pd.concat([train_text_1, test_text_1])

train_text_2 = train['answer']
test_text_2 = test['answer']
all_text_2 = pd.concat([train_text_2, test_text_2])

train_text_3 = train['question_title']
test_text_3 = test['question_title']
all_text_3 = pd.concat([train_text_3, test_text_3])


In [8]:
sample_submission = pd.read_csv('../input/google-quest-challenge/sample_submission.csv').fillna(' ')
sample_submission.head()


,qa_id,question_asker_intent_understanding,question_body_critical,question_conversational,question_expect_short_answer,question_fact_seeking,question_has_commonly_accepted_answer,question_interestingness_others,question_interestingness_self,question_multi_intent,...,question_well_written,answer_helpful,answer_level_of_information,answer_plausible,answer_relevance,answer_satisfaction,answer_type_instructions,answer_type_procedure,answer_type_reason_explanation,answer_well_written
0,6516,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,6168,0.001647,0.001647,0.001647,0.001647,0.001647,0.001647,0.001647,0.001647,0.001647,...,0.001647,0.001647,0.001647,0.001647,0.001647,0.001647,0.001647,0.001647,0.001647,0.001647
2,8575,0.003295,0.003295,0.003295,0.003295,0.003295,0.003295,0.003295,0.003295,0.003295,...,0.003295,0.003295,0.003295,0.003295,0.003295,0.003295,0.003295,0.003295,0.003295,0.003295
3,618,0.004942,0.004942,0.004942,0.004942,0.004942,0.004942,0.004942,0.004942,0.004942,...,0.004942,0.004942,0.004942,0.004942,0.004942,0.004942,0.004942,0.004942,0.004942,0.004942
4,3471,0.006590,0.006590,0.006590,0.006590,0.006590,0.006590,0.006590,0.006590,0.006590,...,0.006590,0.006590,0.006590,0.006590,0.006590,0.006590,0.006590,0.006590,0.006590,0.006590


In [9]:
class_names = list(sample_submission.columns[1:])
class_names


['question_asker_intent_understanding',
 'question_body_critical',
 'question_conversational',
 'question_expect_short_answer',
 'question_fact_seeking',
 'question_has_commonly_accepted_answer',
 'question_interestingness_others',
 'question_interestingness_self',
 'question_multi_intent',
 'question_not_really_a_question',
 'question_opinion_seeking',
 'question_type_choice',
 'question_type_compare',
 'question_type_consequence',
 'question_type_definition',
 'question_type_entity',
 'question_type_instructions',
 'question_type_procedure',
 'question_type_reason_explanation',
 'question_type_spelling',
 'question_well_written',
 'answer_helpful',
 'answer_level_of_information',
 'answer_plausible',
 'answer_relevance',
 'answer_satisfaction',
 'answer_type_instructions',
 'answer_type_procedure',
 'answer_type_reason_explanation',
 'answer_well_written']

In [10]:
class_names_q = class_names[:21]
class_names_a = class_names[21:]
class_names_a


['answer_helpful',
 'answer_level_of_information',
 'answer_plausible',
 'answer_relevance',
 'answer_satisfaction',
 'answer_type_instructions',
 'answer_type_procedure',
 'answer_type_reason_explanation',
 'answer_well_written']

In [11]:
class_names_2 = [class_name+'_2' for class_name in class_names]
for class_name in class_names:
    train[class_name+'_2'] = (train[class_name].values >= 0.5)*1


In [12]:
%%time
word_vectorizer = TfidfVectorizer(
    sublinear_tf=True,
    strip_accents='unicode',
    analyzer='word',
    token_pattern=r'\w{1,}',
    stop_words='english',
    ngram_range=(1, 2),
    max_features=20000)
word_vectorizer.fit(all_text_1)
train_word_features_1 = word_vectorizer.transform(train_text_1)
test_word_features_1 = word_vectorizer.transform(test_text_1)

word_vectorizer = TfidfVectorizer(
    sublinear_tf=True,
    strip_accents='unicode',
    analyzer='word',
    token_pattern=r'\w{1,}',
    stop_words='english',
    ngram_range=(1, 2),
    max_features=20000)
word_vectorizer.fit(all_text_2)
train_word_features_2 = word_vectorizer.transform(train_text_2)
test_word_features_2 = word_vectorizer.transform(test_text_2)

word_vectorizer = TfidfVectorizer(
    sublinear_tf=True,
    strip_accents='unicode',
    analyzer='word',
    token_pattern=r'\w{1,}',
    stop_words='english',
    ngram_range=(1, 2),
    max_features=20000)
word_vectorizer.fit(all_text_3)
train_word_features_3 = word_vectorizer.transform(train_text_3)
test_word_features_3 = word_vectorizer.transform(test_text_3)

char_vectorizer = TfidfVectorizer(
    sublinear_tf=True,
    strip_accents='unicode',
    analyzer='char',
    stop_words='english',
    ngram_range=(1, 4),
    max_features=50000)
char_vectorizer.fit(all_text_1)
train_char_features_1 = char_vectorizer.transform(train_text_1)
test_char_features_1 = char_vectorizer.transform(test_text_1)

char_vectorizer = TfidfVectorizer(
    sublinear_tf=True,
    strip_accents='unicode',
    analyzer='char',
    stop_words='english',
    ngram_range=(1, 4),
    max_features=50000)
char_vectorizer.fit(all_text_2)
train_char_features_2 = char_vectorizer.transform(train_text_2)
test_char_features_2 = char_vectorizer.transform(test_text_2)

char_vectorizer = TfidfVectorizer(
    sublinear_tf=True,
    strip_accents='unicode',
    analyzer='char',
    stop_words='english',
    ngram_range=(1, 4),
    max_features=50000)
char_vectorizer.fit(all_text_3)
train_char_features_3 = char_vectorizer.transform(train_text_3)
test_char_features_3 = char_vectorizer.transform(test_text_3)

train_features_1 = hstack([train_char_features_1, train_word_features_1, train_char_features_3, train_word_features_3])
test_features_1 = hstack([test_char_features_1, test_word_features_1, test_char_features_3, test_word_features_3])
train_features_2 = hstack([train_char_features_2, train_word_features_2])
test_features_2 = hstack([test_char_features_2, test_word_features_2])


/usr/local/lib/python3.11/dist-packages/sklearn/feature_extraction/text.py:550: UserWarning: The parameter 'stop_words' will not be used since 'analyzer' != 'word'
  warnings.warn(


CPU times: user 25.1 s, sys: 556 ms, total: 25.7 s
Wall time: 25.2 s


In [13]:
train_features_1= train_features_1.tocsr()
train_features_2= train_features_2.tocsr()


In [14]:
%%time
submission = pd.DataFrame.from_dict({'qa_id': test['qa_id']})

train_preds = []
test_preds = []
scores = []
spearman_scores = []

for class_name in tqdm_notebook(class_names_q):
    print(class_name)
    Y = train[class_name]
    
    n_splits = 3
    kf = KFold(n_splits=n_splits, random_state=47)

    train_oof_1 = np.zeros((train_features_1.shape[0], ))
    test_preds_1 = 0
    
    score = 0

    for jj, (train_index, val_index) in enumerate(kf.split(train_features_1)):
        #print("Fitting fold", jj+1)
        train_features = train_features_1[train_index]
        train_target = Y[train_index]

        val_features = train_features_1[val_index]
        val_target = Y[val_index]

        model = Ridge(alpha = 20)
        model.fit(train_features, train_target)
        val_pred = model.predict(val_features)
        train_oof_1[val_index] = val_pred
        #print("Fold auc:", roc_auc_score(val_target, val_pred))
        #spearman_corr
        #score += roc_auc_score(val_target, val_pred)/n_splits

        test_preds_1 += model.predict(test_features_1)/n_splits
        del train_features, train_target, val_features, val_target
        gc.collect()
        
    model = Ridge(alpha = 20)
    model.fit(train_features_1, Y)
    
    preds = model.predict(test_features_1)
    mms = MinMaxScaler(copy=True, feature_range=(0, 1))
    preds = mms.fit_transform(preds.reshape(-1, 1)).flatten()
    submission[class_name] = (preds+0.0000005)/1.000001
    spearman_score = spearman_corr(train[class_name], train_oof_1)
    print("spearman_corr:", spearman_score) 
    spearman_scores.append(spearman_score)
    score = roc_auc_score(train[class_name+'_2'], train_oof_1)    
    print("auc:", score, "\n")
    train_preds.append(train_oof_1)
    test_preds.append(test_preds_1)
    scores.append(score)
    
    


<timed exec>:8: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`


  0%|          | 0/21 [00:00<?, ?it/s]

question_asker_intent_understanding


ValueError: Setting a random_state has no effect since shuffle is False. You should leave random_state to its default (None), or set shuffle=True.

In [15]:
%%time

for class_name in tqdm_notebook(class_names_a):
    print(class_name)
    Y = train[class_name]
    
    n_splits = 3
    kf = KFold(n_splits=n_splits, random_state=47)

    train_oof_2 = np.zeros((train_features_2.shape[0], ))
    test_preds_2 = 0
    
    score = 0

    for jj, (train_index, val_index) in enumerate(kf.split(train_features_1)):
        #print("Fitting fold", jj+1)
        train_features = train_features_2[train_index]
        train_target = Y[train_index]

        val_features = train_features_2[val_index]
        val_target = Y[val_index]

        model = Ridge(alpha = 20)
        model.fit(train_features, train_target)
        val_pred = model.predict(val_features)
        train_oof_2[val_index] = val_pred
        #print("Fold auc:", roc_auc_score(val_target, val_pred))
        #score += roc_auc_score(val_target, val_pred)/n_splits

        test_preds_2 += model.predict(test_features_2)/n_splits
        del train_features, train_target, val_features, val_target
        gc.collect()
        
    model = Ridge(alpha = 20)
    model.fit(train_features_2, Y)
    
    preds = model.predict(test_features_2)
    mms = MinMaxScaler(copy=True, feature_range=(0, 1))
    preds = mms.fit_transform(preds.reshape(-1, 1)).flatten()
    submission[class_name] = (preds+0.0000005)/1.000001
        
    score = roc_auc_score(train[class_name+'_2'], train_oof_2) 
    
    
    spearman_score = spearman_corr(train[class_name], train_oof_2)
    print("spearman_corr:", spearman_score)
    print("auc:", score, "\n")
    spearman_scores.append(spearman_score)
    
    train_preds.append(train_oof_2)
    test_preds.append(test_preds_2)
    scores.append(score)


<timed exec>:1: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`


  0%|          | 0/9 [00:00<?, ?it/s]

answer_helpful


ValueError: Setting a random_state has no effect since shuffle is False. You should leave random_state to its default (None), or set shuffle=True.

In [16]:
print("Mean auc:", np.mean(scores))
print("Mean spearman_scores", np.mean(spearman_scores))


Mean auc: nan
Mean spearman_scores nan


/usr/local/lib/python3.11/dist-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.11/dist-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [17]:
submission.to_csv('submission.csv', index=False)
submission.head()


,qa_id
0,6516
1,6168
2,8575
3,618
4,3471


In [18]:
submission[class_names].values.max()


KeyError: "None of [Index(['question_asker_intent_understanding', 'question_body_critical',\n       'question_conversational', 'question_expect_short_answer',\n       'question_fact_seeking', 'question_has_commonly_accepted_answer',\n       'question_interestingness_others', 'question_interestingness_self',\n       'question_multi_intent', 'question_not_really_a_question',\n       'question_opinion_seeking', 'question_type_choice',\n       'question_type_compare', 'question_type_consequence',\n       'question_type_definition', 'question_type_entity',\n       'question_type_instructions', 'question_type_procedure',\n       'question_type_reason_explanation', 'question_type_spelling',\n       'question_well_written', 'answer_helpful',\n       'answer_level_of_information', 'answer_plausible', 'answer_relevance',\n       'answer_satisfaction', 'answer_type_instructions',\n       'answer_type_procedure', 'answer_type_reason_explanation',\n       'answer_well_written'],\n      dtype='objec

In [19]:
submission[class_names].values.min()


KeyError: "None of [Index(['question_asker_intent_understanding', 'question_body_critical',\n       'question_conversational', 'question_expect_short_answer',\n       'question_fact_seeking', 'question_has_commonly_accepted_answer',\n       'question_interestingness_others', 'question_interestingness_self',\n       'question_multi_intent', 'question_not_really_a_question',\n       'question_opinion_seeking', 'question_type_choice',\n       'question_type_compare', 'question_type_consequence',\n       'question_type_definition', 'question_type_entity',\n       'question_type_instructions', 'question_type_procedure',\n       'question_type_reason_explanation', 'question_type_spelling',\n       'question_well_written', 'answer_helpful',\n       'answer_level_of_information', 'answer_plausible', 'answer_relevance',\n       'answer_satisfaction', 'answer_type_instructions',\n       'answer_type_procedure', 'answer_type_reason_explanation',\n       'answer_well_written'],\n      dtype='objec